# Experiment - HotpotQA
Cross-dataset replication: 3 models × 3 prompt types × {r=0, r=1} on HotpotQA (N=50).
Results saved to `results/hotpotqa_results.csv`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import yaml
import pandas as pd
from pathlib import Path

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

MODELS       = cfg["models"]["available"]
PROMPT_TYPES = ["standard_qa", "cot_qa", "vigilant_qa"]
N            = cfg["evaluation"]["n_examples"]
SEED         = cfg["seed"]
RESULTS_DIR  = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Models: {MODELS}")
print(f"Prompt types: {PROMPT_TYPES}")
print(f"N: {N}  seed: {SEED}  k: {cfg['retrieval']['k']}")

Models: ['Qwen/Qwen3.5-2B', 'google/gemma-4-E2B-it']
Prompt types: ['standard_qa', 'cot_qa', 'vigilant_qa']
N: 75  seed: 42  k: 10


In [2]:
from src.data.download_hotpotqa import download
from src.data.hotpotqa_loader import load_hotpotqa

dev_path = Path("..") / cfg["dataset"]["hotpotqa_dev"]
download(target=dev_path)
POOL_SIZE = 200  # fixed pool so cache stays valid when N changes
all_clean = load_hotpotqa(str(dev_path), max_examples=POOL_SIZE)
examples = all_clean[:N]  # same Python objects — identity-based exclusion works
print(f"Pool: {len(all_clean)}  Evaluated: {len(examples)} HotpotQA examples")

Pool: 200  Evaluated: 75 HotpotQA examples


In [3]:
from src.retrieval.embedder import Embedder
from src.retrieval.retriever import Retriever

emb_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["embeddings_subdir"])
embedder = Embedder(model_name=cfg["retrieval"]["embedding_model"], cache_dir=emb_cache)
retriever = Retriever(embedder=embedder, k=cfg["retrieval"]["k"])
print(f"k={cfg['retrieval']['k']}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

k=10


In [4]:
from src.data.hotpotqa_loader import load_hotpotqa
from src.evaluation import qa_scorer
from src.generation.llm_client import HuggingFaceClient

all_poisoned = load_hotpotqa("../" + cfg["dataset"]["hotpotqa_dev_poisoned"])
examples_poisoned = all_poisoned[:N]  # same objects — identity exclusion works
print(f"Clean examples   : {len(examples)}")
print(f"Poisoned examples: {len(examples_poisoned)}")

records = []

for model in MODELS:
    print(f"\n{'='*60}\nModel: {model}\n{'='*60}")
    llm_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["llm_subdir"])
    llm = HuggingFaceClient(
        model=model,
        temperature=cfg["models"]["temperature"],
        cache_dir=llm_cache,
    )
    with embedder, llm:
        for prompt_type in PROMPT_TYPES:
            print(f"  prompt={prompt_type}  r=0", end="")
            m0 = qa_scorer.run(
                examples=examples, retriever=retriever, llm=llm,
                prompt_type=prompt_type, seed=SEED,
                full_dataset=all_clean,
            )
            records.append({"model": model, "prompt_type": prompt_type,
                            "poison_rate": 0, **m0})
            print(f" EM={m0['exact_match']:.3f}  r=1", end="")
            m1 = qa_scorer.run(
                examples=examples_poisoned, retriever=retriever, llm=llm,
                prompt_type=prompt_type, seed=SEED,
                full_dataset=all_poisoned,
            )
            records.append({"model": model, "prompt_type": prompt_type,
                            "poison_rate": 1, **m1})
            print(f" EM={m1['exact_match']:.3f}")

print("\nSweep complete.")

Clean examples   : 75
Poisoned examples: 75

Model: Qwen/Qwen3.5-2B
  prompt=standard_qa  r=0 EM=0.573  r=1 EM=0.507
  prompt=cot_qa  r=0 EM=0.520  r=1 EM=0.440
  prompt=vigilant_qa  r=0 EM=0.307  r=1 EM=0.307

Model: google/gemma-4-E2B-it
  prompt=standard_qa  r=0 EM=0.467  r=1 EM=0.373
  prompt=cot_qa  r=0 EM=0.307  r=1 EM=0.307
  prompt=vigilant_qa  r=0 EM=0.307  r=1 EM=0.280

Sweep complete.


In [5]:
df = pd.DataFrame(records)
out_path = RESULTS_DIR / "hotpotqa_results.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} rows to {out_path}")
df

Saved 12 rows to ../results/hotpotqa_results.csv


,model,prompt_type,poison_rate,exact_match,token_f1,hallucination_rate
0,Qwen/Qwen3.5-2B,standard_qa,0,0.573333,0.684413,0.133333
1,Qwen/Qwen3.5-2B,standard_qa,1,0.506667,0.605746,0.133333
2,Qwen/Qwen3.5-2B,cot_qa,0,0.520000,0.655232,0.146667
3,Qwen/Qwen3.5-2B,cot_qa,1,0.440000,0.567514,0.133333
4,Qwen/Qwen3.5-2B,vigilant_qa,0,0.306667,0.456988,0.186667
5,Qwen/Qwen3.5-2B,vigilant_qa,1,0.306667,0.403103,0.160000
6,google/gemma-4-E2B-it,standard_qa,0,0.466667,0.580054,0.053333
7,google/gemma-4-E2B-it,standard_qa,1,0.373333,0.536498,0.093333
8,google/gemma-4-E2B-it,cot_qa,0,0.306667,0.465007,0.053333
9,google/gemma-4-E2B-it,cot_qa,1,0.306667,0.423224,0.080000
